# Paired grading reconciliation
Aggregate checks only; no private questions or answers. Invalid judge outputs are excluded for all models on the same item, never scored as incorrect. Maintenance is in-training.


In [ ]:
import json
from pathlib import Path
p=Path('docs') if Path('docs').is_dir() else Path('.')
s=json.loads((p/'targeted_book_pilot_grading_20260908.safe.json').read_text(encoding='utf-8'))
assert s['items']==sum(s['status'].values())==55
assert s['status']=={'scored':50,'invalid_review':5}
d=s['cohorts']['development']; m=s['cohorts']['in_training_maintenance']
assert (d['items'],d['scored'],m['items'],m['scored'])==(39,34,16,16)
assert [d['metrics'][k]['closed']['both_pass'] for k in ('baseline','step5','step10')]==[7,7,3]
for cohort in (d,m):
    for model,counts in cohort['paired_closed_both'].items():
        assert sum(counts.values())==cohort['scored']
        assert counts.get('gain',0)-counts.get('loss',0)==cohort['metrics'][model]['closed']['both_pass']-cohort['metrics']['baseline']['closed']['both_pass']
    for metrics in cohort['metrics'].values():
        for mode in metrics.values():
            assert mode['answer_count']==2*cohort['scored']
            assert mode['both_pass']*2<=mode['answer_pass']<=mode['answer_count']
assert s['usage']['total_tokens']==s['usage']['prompt_tokens']+s['usage']['completion_tokens']==407209
print('PASS: common denominators, paired changes, separate cohorts and returned usage.')
